In [2]:
%pip install selenium webdriver-manager

  Using cached selenium-4.41.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached trio-0.33.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached wsproto-1.3.2-py3-non


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import time
import json
import re  # 用於提取總評論數中的數字
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from tqdm import tqdm

# 設定目標
HOTEL_NAME = "路境行旅 府前館 Finders Hotel Fuqian" 
HOTEL_URL = "https://www.agoda.com/zh-tw/finders-hotel-fu-qian/reviews/taipei-tw.html"
MAX_REVIEWS = 5000 # 抓取上限

def get_total_reviews(driver):
    """從頁面中定位並提取總評論數"""
    try:
        # 1. 嘗試多種可能的定位器 (CSS selector 或 XPath)
        # 第一個是你截圖中的, 第二個是備用的常見結構
        selectors = [
            "span.Review__SummaryContainer--left",
            "//span[contains(text(), '真實住客評價')]",
            ".Review-filter-sectionTitle"
        ]
        
        element = None
        for sel in selectors:
            try:
                if sel.startswith("//"):
                    element = WebDriverWait(driver, 7).until(
                        EC.presence_of_element_located((By.XPATH, sel))
                    )
                else:
                    element = WebDriverWait(driver, 7).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, sel))
                    )
                if element: break
            except:
                continue

        if not element:
            print("⚠️ 警告：找不到評論數元素")
            return None

        text = element.text
        print(f"🔍 偵測到評論文字: {text}")

        # 2. 提取數字 (1,818 -> 1818)
        numbers = re.findall(r'\d+(?:,\d+)*', text)
        if numbers:
            total = int(numbers[0].replace(',', ''))
            return total
            
    except Exception as e:
        print(f"❌ 取得總評論數時發生異常: {e}")
    return None

def run_agoda_spider(url, max_target):
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    all_results = []
    seen_ids = set()

    try:
        driver.get(url)
        wait = WebDriverWait(driver, 20)
        
        # 1. 先滑動一點點，確保評論區塊被觸發載入
        driver.execute_script("window.scrollBy(0, 500);")
        time.sleep(2) 

        # 2. 獲取總數
        total_available = get_total_reviews(driver)
        
        # --- [修正點]：如果第一次沒抓到，給它第二次機會 ---
        if not total_available:
            print("🔄 第一次嘗試未偵測到總數，嘗試深度捲動後補抓...")
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(3)
            total_available = get_total_reviews(driver)

        if total_available:
            print(f"📊 偵測到飯店實際評論總數: {total_available}")
            target_count = min(max_target, total_available)
        else:
            print("⚠️ 警告：最終仍無法取得總數，將使用預設上限。")
            target_count = max_target
        
        print(f"🎯 最終抓取目標設定為: {target_count} 則\n")

        while len(all_results) < target_count:
            # 顯示進度百分比
            percent = (len(all_results) / target_count) * 100
            print(f"📜 進度: {percent:.1f}% | 已抓取: {len(all_results)}/{target_count}...")
            
            # 深度滾動確保元素加載
            for _ in range(5):
                driver.execute_script("window.scrollBy(0, 1000);")
                time.sleep(0.5)
            
            card_selector = '[data-element-name="review-comment"]'
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, card_selector)))
            cards = driver.find_elements(By.CSS_SELECTOR, card_selector)

            for card in cards:
                if len(all_results) >= target_count: break
                
                try:
                    r_id = card.get_attribute("data-review-id")
                    if not r_id or r_id in seen_ids: continue
                    
                    score_el = card.find_elements(By.CSS_SELECTOR, ".Review-comment-leftScore")
                    if not score_el: continue

                    # 展開「閱讀更多」
                    try:
                        read_more_btn = card.find_elements(By.CSS_SELECTOR, '[data-element-name="review-read-more-button"]')
                        if read_more_btn:
                            driver.execute_script("arguments[0].click();", read_more_btn[0])
                            time.sleep(0.3)
                    except: pass

                    # 提取基本資料
                    score = score_el[0].text
                    content = card.find_element(By.CSS_SELECTOR, ".Review-comment-bodyText").text.strip()
                    
                    # 處理日期
                    comment_date = "未知日期"
                    all_spans = card.find_elements(By.TAG_NAME, "span")
                    for s in all_spans:
                        txt = s.text
                        if "202" in txt and ("評價" in txt or "發表" in txt):
                            comment_date = txt
                            break

                    # 提取「房型」與「旅遊類型」
                    room_type = ""
                    travel_type = ""
                    try:
                        detail_info = card.find_elements(By.CSS_SELECTOR, '.Review-comment-reviewer-subInfo .Review-comment-reviewer-subInfo-item')
                        if len(detail_info) >= 1: travel_type = detail_info[0].text
                        if len(detail_info) >= 2: room_type = detail_info[1].text
                    except: pass

                    # 抓取照片數量
                    photo_count = 0
                    try:
                        photo_buttons = card.find_elements(By.CSS_SELECTOR, 'button[data-element-name="review-comment-ugc-thumbnail"]')
                        photo_count = len(photo_buttons)
                    except: pass

                    # 封裝 JSON
                    review_item = {
                        "飯店名稱": HOTEL_NAME,
                        "評論ID": r_id,
                        "評分": score,
                        "評論內容": content,
                        "評論日期": comment_date,
                        "旅遊類型": travel_type,
                        "房型": room_type,
                        "照片數量": photo_count,
                        "飯店評論總數": total_available if total_available else "未知",
                    }

                    seen_ids.add(r_id)
                    all_results.append(review_item)
                    #print(f"✅ ID: {r_id} | ⭐: {score} | 📷: {photo_count}")

                except Exception:
                    continue

            if len(all_results) >= target_count: break

            # 翻頁邏輯：點擊「顯示更多評價」
            try:
                load_more_btn = driver.find_elements(By.CSS_SELECTOR, '.Review-paginator-button')
                
                if load_more_btn and load_more_btn[0].is_displayed():
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more_btn[0])
                    time.sleep(1)
                    driver.execute_script("arguments[0].click();", load_more_btn[0])
                    # 等待新評論加載
                    time.sleep(3) 
                else:
                    print("✨ 已無更多評論可載入。")
                    break
            except Exception as e:
                print(f"🛑 載入按鈕處理異常: {e}")
                break

        # --- 導出 JSON 檔案 ---
        # --- 改良版：導出與合併 JSON 檔案 ---
        # --- 改良版：按飯店名稱分類並獨立編號 ---
        file_name = "agoda_reviews_organized.json"
        all_data_dict = {}

        # 1. 讀取現有檔案 (字典格式)
        try:
            with open(file_name, "r", encoding="utf-8") as f:
                all_data_dict = json.load(f)
                if not isinstance(all_data_dict, dict):
                    all_data_dict = {}
        except (FileNotFoundError, json.JSONDecodeError):
            all_data_dict = {}

        # 2. 獲取當前飯店已有的資料與 ID
        hotel_data = all_data_dict.get(HOTEL_NAME, [])
        existing_ids = {item["評論ID"] for item in hotel_data}
        
        # 3. 處理本次抓取的資料 (去重並給予獨立編號)
        new_count = len(hotel_data)
        for item in all_results:
            if item["評論ID"] not in existing_ids:
                new_count += 1
                item["編號"] = new_count  # 根據該飯店現有數量編號
                hotel_data.append(item)

        # 4. 更新回總字典並存檔
        all_data_dict[HOTEL_NAME] = hotel_data
        
        with open(file_name, "w", encoding="utf-8") as f:
            json.dump(all_data_dict, f, ensure_ascii=False, indent=2)

        print("\n" + "★" * 20 + " 最終報告 " + "★" * 20)
        print(f"🏨 飯店名稱: {HOTEL_NAME}")
        print(f"✅ 本次新增: {len(all_results) - (new_count - len(hotel_data))} 則 (去重後)")
        print(f"📈 該飯店總計: {len(hotel_data)} 則")
        print(f"📂 數據已更新至: {file_name}")

    except Exception as e:
        print(f"❌ 執行出錯: {e}")
    finally:
        if 'driver' in locals():
            input("\n確認完畢請按 Enter 關閉瀏覽器...")
            driver.quit()

if __name__ == "__main__":
    run_agoda_spider(HOTEL_URL, MAX_REVIEWS)

🔍 偵測到評論文字: 
🔄 第一次嘗試未偵測到總數，嘗試深度捲動後補抓...
🔍 偵測到評論文字: 
⚠️ 警告：最終仍無法取得總數，將使用預設上限。
🎯 最終抓取目標設定為: 5000 則

📜 進度: 0.0% | 已抓取: 0/5000...
📜 進度: 1.0% | 已抓取: 50/5000...
📜 進度: 2.4% | 已抓取: 120/5000...
📜 進度: 3.8% | 已抓取: 190/5000...
📜 進度: 5.2% | 已抓取: 260/5000...
📜 進度: 6.6% | 已抓取: 330/5000...
📜 進度: 8.0% | 已抓取: 400/5000...
📜 進度: 9.4% | 已抓取: 470/5000...
📜 進度: 10.8% | 已抓取: 540/5000...
📜 進度: 12.2% | 已抓取: 610/5000...
📜 進度: 13.6% | 已抓取: 680/5000...
📜 進度: 15.0% | 已抓取: 750/5000...
📜 進度: 16.4% | 已抓取: 820/5000...
📜 進度: 17.8% | 已抓取: 890/5000...
📜 進度: 19.2% | 已抓取: 960/5000...
📜 進度: 20.6% | 已抓取: 1030/5000...
📜 進度: 22.0% | 已抓取: 1100/5000...
📜 進度: 23.4% | 已抓取: 1170/5000...
📜 進度: 24.8% | 已抓取: 1240/5000...
📜 進度: 26.2% | 已抓取: 1310/5000...
📜 進度: 27.6% | 已抓取: 1380/5000...
📜 進度: 29.0% | 已抓取: 1450/5000...
📜 進度: 30.4% | 已抓取: 1520/5000...
📜 進度: 31.8% | 已抓取: 1590/5000...
📜 進度: 33.2% | 已抓取: 1660/5000...
📜 進度: 34.6% | 已抓取: 1730/5000...
📜 進度: 36.0% | 已抓取: 1800/5000...
📜 進度: 37.4% | 已抓取: 1870/5000...
📜 進度: 38.8% | 已抓取: 1939/5000...
📜